In [41]:
from pathlib import Path
import time
import joblib

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [42]:
DATA_PATH = Path(
    "../data/processed/hdb_resale_clean.csv"
)

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["month"]
)

print(
    "Rows:",
    len(df)
)

print(
    "Date range:",
    df["month"].min(),
    "to",
    df["month"].max()
)

Rows: 244230
Date range: 2017-01-01 00:00:00 to 2026-11-01 00:00:00


In [43]:
LATEST_COMPLETE_MONTH = pd.Timestamp(
    "2026-10-01"
)

df_complete = df[
    df["month"]
    <= LATEST_COMPLETE_MONTH
].copy()

print(
    "Complete-month rows:",
    len(df_complete)
)

print(
    "Latest complete month:",
    df_complete["month"].max()
)

Complete-month rows: 243872
Latest complete month: 2026-10-01 00:00:00


In [44]:
train_df = df_complete[
    df_complete["month"]
    <= "2026-06-01"
].copy()

validation_df = df_complete[
    (
        df_complete["month"]
        >= "2026-07-01"
    )
    &
    (
        df_complete["month"]
        <= "2026-08-01"
    )
].copy()

calibration_df = df_complete[
    df_complete["month"]
    == "2026-09-01"
].copy()

final_df = df_complete[
    df_complete["month"]
    == "2026-10-01"
].copy()

print(
    "Train:",
    len(train_df)
)

print(
    "Validation:",
    len(validation_df)
)

print(
    "Calibration:",
    len(calibration_df)
)

print(
    "Final:",
    len(final_df)
)

Train: 234292
Validation: 5166
Calibration: 2239
Final: 2175


In [45]:
print(
    "Train:",
    train_df["month"].min(),
    "to",
    train_df["month"].max()
)

print(
    "Validation:",
    validation_df["month"].min(),
    "to",
    validation_df["month"].max()
)

print(
    "Calibration:",
    calibration_df["month"].min(),
    "to",
    calibration_df["month"].max()
)

print(
    "Final:",
    final_df["month"].min(),
    "to",
    final_df["month"].max()
)

Train: 2017-01-01 00:00:00 to 2026-06-01 00:00:00
Validation: 2026-07-01 00:00:00 to 2026-08-01 00:00:00
Calibration: 2026-09-01 00:00:00 to 2026-09-01 00:00:00
Final: 2026-10-01 00:00:00 to 2026-10-01 00:00:00


In [46]:
FEATURES = [
    "town",
    "flat_type",
    "storey_mid",
    "floor_area_sqm",
    "flat_model",
    "lease_commence_date",
    "remaining_lease_months",
    "month_index"
]

TARGET = "resale_price"

X_train = train_df[
    FEATURES
].copy()

y_train = train_df[
    TARGET
].copy()

X_validation = validation_df[
    FEATURES
].copy()

y_validation = validation_df[
    TARGET
].copy()

print(
    "X_train:",
    X_train.shape
)

print(
    "X_validation:",
    X_validation.shape
)

X_train: (234292, 8)
X_validation: (5166, 8)


In [47]:
baseline_lookup = (
    train_df
    .groupby(
        [
            "town",
            "flat_type"
        ]
    )["resale_price"]
    .median()
)

global_median = (
    train_df[
        "resale_price"
    ]
    .median()
)


def baseline_predict(row):

    key = (
        row["town"],
        row["flat_type"]
    )

    return baseline_lookup.get(
        key,
        global_median
    )


baseline_predictions = (
    validation_df
    .apply(
        baseline_predict,
        axis=1
    )
)

baseline_mae = mean_absolute_error(
    y_validation,
    baseline_predictions
)

baseline_ape = (
    np.abs(
        y_validation.values
        - baseline_predictions.values
    )
    / y_validation.values
)

baseline_median_ape = (
    np.median(
        baseline_ape
    )
    * 100
)

print(
    "Baseline MAE:",
    round(
        baseline_mae,
        2
    )
)

print(
    "Baseline Median APE:",
    round(
        baseline_median_ape,
        4
    ),
    "%"
)

Baseline MAE: 149637.42
Baseline Median APE: 19.2476 %


In [48]:
categorical_features = [
    "town",
    "flat_type",
    "flat_model"
]

numeric_features = [
    "storey_mid",
    "floor_area_sqm",
    "lease_commence_date",
    "remaining_lease_months",
    "month_index"
]

In [49]:
sparse_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

In [50]:
dense_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

In [51]:
models = {
    "XGBoost": Pipeline(
        steps=[
            (
                "preprocessor",
                sparse_preprocessor
            ),
            (
                "model",
                XGBRegressor(
                    n_estimators=500,
                    learning_rate=0.05,
                    max_depth=8,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="reg:squarederror",
                    n_jobs=-1,
                    random_state=42
                )
            )
        ]
    ),

    "LightGBM": Pipeline(
        steps=[
            (
                "preprocessor",
                sparse_preprocessor
            ),
            (
                "model",
                LGBMRegressor(
                    n_estimators=500,
                    learning_rate=0.05,
                    num_leaves=31,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    random_state=42,
                    n_jobs=-1,
                    verbosity=-1
                )
            )
        ]
    ),

    "CatBoost": Pipeline(
        steps=[
            (
                "preprocessor",
                sparse_preprocessor
            ),
            (
                "model",
                CatBoostRegressor(
                    iterations=500,
                    learning_rate=0.05,
                    depth=8,
                    loss_function="MAE",
                    random_seed=42,
                    verbose=False,
                    thread_count=-1
                )
            )
        ]
    ),

    "HistGradientBoosting": Pipeline(
        steps=[
            (
                "preprocessor",
                dense_preprocessor
            ),
            (
                "model",
                HistGradientBoostingRegressor(
                    max_iter=200,
                    random_state=42
                )
            )
        ]
    )
    
}

In [52]:
def evaluate_model(
    model_name,
    model,
    X_train,
    y_train,
    X_validation,
    y_validation,
    model_dir="../models/model_selection"
):
    """
    Train and evaluate one candidate model.
    """

    model_dir = Path(
        model_dir
    )

    model_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    start_time = (
        time.perf_counter()
    )

    model.fit(
        X_train,
        y_train
    )

    training_time = (
        time.perf_counter()
        - start_time
    )

    predictions = model.predict(
        X_validation
    )

    mae = mean_absolute_error(
        y_validation,
        predictions
    )

    ape = (
        np.abs(
            y_validation.values
            - predictions
        )
        / y_validation.values
    )

    median_ape = (
        np.median(
            ape
        )
        * 100
    )

    model_path = (
        model_dir
        / f"{model_name}.joblib"
    )

    joblib.dump(
        model,
        model_path,
        compress=3
    )

    model_size_mb = (
        model_path.stat().st_size
        / (1024 ** 2)
    )

    return {
        "model": model_name,
        "mae": mae,
        "median_ape_pct": median_ape,
        "training_time_sec": training_time,
        "model_size_mb": model_size_mb
    }

In [53]:
results = []

for model_name, model in models.items():

    print(
        f"Training {model_name}..."
    )

    result = evaluate_model(
        model_name=model_name,
        model=model,
        X_train=X_train,
        y_train=y_train,
        X_validation=X_validation,
        y_validation=y_validation
    )

    results.append(
        result
    )

    print(
        f"MAE: "
        f"{result['mae']:,.2f}"
    )

    print(
        f"Median APE: "
        f"{result['median_ape_pct']:.4f}%"
    )

    print(
        f"Training time: "
        f"{result['training_time_sec']:.2f}s"
    )

    print(
        f"Model size: "
        f"{result['model_size_mb']:.2f} MB"
    )

    print()

Training XGBoost...
MAE: 30,528.76
Median APE: 3.6136%
Training time: 3.29s
Model size: 2.29 MB

Training LightGBM...
MAE: 35,106.47
Median APE: 4.1757%
Training time: 1.84s
Model size: 0.48 MB

Training CatBoost...
MAE: 37,387.52
Median APE: 4.1429%
Training time: 8.39s
Model size: 0.37 MB

Training HistGradientBoosting...
MAE: 36,875.07
Median APE: 4.3046%
Training time: 3.74s
Model size: 0.30 MB



In [54]:
results_df = (
    pd.DataFrame(
        results
    )
    .sort_values(
        "mae"
    )
    .reset_index(
        drop=True
    )
)

results_df

,model,mae,median_ape_pct,training_time_sec,model_size_mb
0,XGBoost,30528.756904,3.613579,3.285136,2.289689
1,LightGBM,35106.467726,4.175746,1.837691,0.483354
2,HistGradientBoosting,36875.074439,4.304610,3.737078,0.300574
3,CatBoost,37387.515315,4.142862,8.387838,0.369706


In [55]:
baseline_row = pd.DataFrame(
    [
        {
            "model": "Simple Baseline",
            "mae": baseline_mae,
            "median_ape_pct": baseline_median_ape,
            "training_time_sec": 0,
            "model_size_mb": 0
        }
    ]
)

comparison_df = pd.concat(
    [
        results_df,
        baseline_row
    ],
    ignore_index=True
)

comparison_df

,model,mae,median_ape_pct,training_time_sec,model_size_mb
0,XGBoost,30528.756904,3.613579,3.285136,2.289689
1,LightGBM,35106.467726,4.175746,1.837691,0.483354
2,HistGradientBoosting,36875.074439,4.304610,3.737078,0.300574
3,CatBoost,37387.515315,4.142862,8.387838,0.369706
4,Simple Baseline,149637.420031,19.247649,0.000000,0.000000


In [56]:
SELECTED_MODEL = "XGBoost"

selected_result = results_df[
    results_df["model"]
    == SELECTED_MODEL
].iloc[0]

selected_result

model                     XGBoost
mae                  30528.756904
median_ape_pct           3.613579
training_time_sec        3.285136
model_size_mb            2.289689
Name: 0, dtype: object

In [57]:
SELECTED_MODEL_PATH = Path(
    "../models/model_selection/XGBoost.joblib"
)

selected_model = joblib.load(
    SELECTED_MODEL_PATH
)

print(
    "Selected model:",
    SELECTED_MODEL
)

print(
    "Model path:",
    SELECTED_MODEL_PATH
)

Selected model: XGBoost
Model path: ..\models\model_selection\XGBoost.joblib


In [58]:
FINAL_MODEL_DIR = Path(
    "../models/final"
)

FINAL_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_MODEL_PATH = (
    FINAL_MODEL_DIR
    / "hdb_resale_xgboost.joblib"
)

joblib.dump(
    selected_model,
    FINAL_MODEL_PATH,
    compress=3
)

print(
    "Saved final model to:",
    FINAL_MODEL_PATH
)

Saved final model to: ..\models\final\hdb_resale_xgboost.joblib


In [59]:
model_metadata = {
    "model_name": SELECTED_MODEL,
    "train_start": str(
        train_df["month"].min().date()
    ),
    "train_end": str(
        train_df["month"].max().date()
    ),
    "validation_start": str(
        validation_df["month"].min().date()
    ),
    "validation_end": str(
        validation_df["month"].max().date()
    ),
    "validation_mae": float(
        selected_result["mae"]
    ),
    "validation_median_ape_pct": float(
        selected_result[
            "median_ape_pct"
        ]
    ),
    "training_time_sec": float(
        selected_result[
            "training_time_sec"
        ]
    ),
    "model_size_mb": float(
        selected_result[
            "model_size_mb"
        ]
    ),
    "features": FEATURES
}

model_metadata

{'model_name': 'XGBoost',
 'train_start': '2017-01-01',
 'train_end': '2026-06-01',
 'validation_start': '2026-07-01',
 'validation_end': '2026-08-01',
 'validation_mae': 30528.756904156988,
 'validation_median_ape_pct': 3.613578907203907,
 'training_time_sec': 3.2851364000234753,
 'model_size_mb': 2.289689064025879,
 'features': ['town',
  'flat_type',
  'storey_mid',
  'floor_area_sqm',
  'flat_model',
  'lease_commence_date',
  'remaining_lease_months',
  'month_index']}

In [60]:
import json

METADATA_PATH = (
    FINAL_MODEL_DIR
    / "hdb_resale_xgboost_metadata.json"
)

with open(
    METADATA_PATH,
    "w"
) as file:

    json.dump(
        model_metadata,
        file,
        indent=4
    )

print(
    "Saved metadata to:",
    METADATA_PATH
)

Saved metadata to: ..\models\final\hdb_resale_xgboost_metadata.json


In [61]:
RESULTS_PATH = (
    FINAL_MODEL_DIR
    / "model_selection_results.csv"
)

comparison_df.to_csv(
    RESULTS_PATH,
    index=False
)

print(
    "Saved model comparison to:",
    RESULTS_PATH
)

Saved model comparison to: ..\models\final\model_selection_results.csv
